# Hackathon 7.0 — MobileFaceNet on Kaggle (CASIA-WebFace `.rec`)
### Trains the offline face-recognition model, evaluates on real LFW, exports constraint-satisfying weights

This version reads the **MXNet RecordIO** dataset you attached (`train.rec` / `train.idx` + `eval/*.bin`) with a
pure-Python reader — **no `mxnet` install needed** — and evaluates on the bundled **`lfw.bin`** for a real LFW number.

**Settings before you run:**
1. **Accelerator → GPU T4 x2** (pick **T4**, *not* P100 — your installed PyTorch build does not support the P100's
   compute capability 6.0 and CUDA kernels will fail on it).
2. **Internet → On** (for `pip` / the optional TFLite step).
3. The CASIA-WebFace `.rec` dataset is already attached (the cells auto-locate `train.rec` and `lfw.bin`).

**Dataset layout (for reference, auto-detected):**
```
.../casia-webface/  property  train.idx  train.lst  train.rec
.../eval/           lfw.bin  cfp_fp.bin  agedb_30.bin  calfw.bin  cplfw.bin ...
```

**Outputs** in `/kaggle/working/artifacts/`: `mobilefacenet_fp32.pt`, `*_fp32.onnx`, `*_int8.onnx`, optional `*.tflite`.
**Verified constraints:** model < 20 MB · CPU latency < 1 s/face · LFW accuracy reported.

## 0 · GPU check & dependencies

In [1]:
import subprocess, sys
def pip(*p): subprocess.run([sys.executable,"-m","pip","install","-q",*p], check=False)
pip("onnx","onnxruntime","onnxscript","scikit-learn")

import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Enable GPU: Settings -> Accelerator -> GPU T4 x2 (avoid P100)."
if "P100" in (torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""):
    print("WARNING: P100 detected — its compute capability (6.0) may be unsupported by this torch build. "
          "Switch Accelerator to 'GPU T4 x2' if training throws CUDA errors.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 13.8 MB/s eta 0:00:00
torch: 2.10.0+cu128 | CUDA: True | Tesla T4


## 1 · Configuration

In [2]:
import os, time, math, random, numpy as np
CFG = dict(
    IMG_SIZE=112, EMB_SIZE=128,
    BATCH=256, MAX_EPOCHS=40,
    LR=0.1, WEIGHT_DECAY=5e-4, MOMENTUM=0.9,
    ARC_S=64.0, ARC_M=0.50, WARMUP_ITERS=500,
    TIME_BUDGET_HOURS=8.0,      # Kaggle GPU session cap ~12h; leave margin for eval+export
    SEED=42, OUT_DIR="/kaggle/working/artifacts",
)
os.makedirs(CFG["OUT_DIR"], exist_ok=True)
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"])
DEVICE="cuda"; print(CFG)


{'IMG_SIZE': 112, 'EMB_SIZE': 128, 'BATCH': 256, 'MAX_EPOCHS': 40, 'LR': 0.1, 'WEIGHT_DECAY': 0.0005, 'MOMENTUM': 0.9, 'ARC_S': 64.0, 'ARC_M': 0.5, 'WARMUP_ITERS': 500, 'TIME_BUDGET_HOURS': 8.0, 'SEED': 42, 'OUT_DIR': '/kaggle/working/artifacts'}


## 2 · Locate `train.rec` and `lfw.bin`
Globs the attached dataset wherever Kaggle mounted it, so the exact mount path doesn't matter.

In [3]:
import glob
recs = glob.glob("/kaggle/input/**/train.rec", recursive=True)
assert recs, "train.rec not found under /kaggle/input — is the CASIA-WebFace .rec dataset attached?"
REC = recs[0]; IDX = REC[:-4] + ".idx"
assert os.path.exists(IDX), f"train.idx missing next to {REC}"
lfws = glob.glob("/kaggle/input/**/lfw.bin", recursive=True)
LFW_BIN = lfws[0] if lfws else None
print("REC:", REC); print("IDX:", IDX); print("LFW_BIN:", LFW_BIN)


REC: /kaggle/input/datasets/debarghamitraroy/casia-webface/casia-webface/train.rec
IDX: /kaggle/input/datasets/debarghamitraroy/casia-webface/casia-webface/train.idx
LFW_BIN: /kaggle/input/datasets/debarghamitraroy/casia-webface/eval/lfw.bin


## 3 · Pure-Python RecordIO reader + dataset
Parses the MXNet RecordIO framing and IRHeader directly. Labels are read from record headers only (24 bytes each,
image bytes skipped) so dataset init is fast. Per-process file handles make it safe with DataLoader workers.

In [4]:
import struct, io
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

_MAGIC = 0xced7230a
_IR = "<IfQQ"; _IRN = struct.calcsize(_IR)   # 24-byte IRHeader: flag,label,id,id2

class RecReader:
    def __init__(self, rec_path, idx_path):
        self.rec_path = rec_path; self.offsets = {}
        with open(idx_path) as f:
            for line in f:
                k, v = line.strip().split("\t"); self.offsets[int(k)] = int(v)
        self.f = None; self.pid = None
    def _h(self):
        if self.f is None or self.pid != os.getpid():
            self.f = open(self.rec_path, "rb"); self.pid = os.getpid()
        return self.f
    def _seek(self, i):
        f = self._h(); f.seek(self.offsets[i])
        assert struct.unpack("<I", f.read(4))[0] == _MAGIC, "bad RecordIO magic"
        length = struct.unpack("<I", f.read(4))[0] & ((1 << 29) - 1)
        return f, length
    def header_full(self, i):                 # full header (+ label array) + remaining bytes
        f, length = self._seek(i); blob = f.read(length)
        flag, label, _, _ = struct.unpack(_IR, blob[:_IRN]); rest = blob[_IRN:]
        if flag > 0:
            label = list(struct.unpack("<%df" % flag, rest[:4*flag])); rest = rest[4*flag:]
        return flag, label, rest
    def label_only(self, i):                  # fast: read 24-byte header, skip image
        f, _ = self._seek(i)
        flag, label, _, _ = struct.unpack(_IR, f.read(_IRN)); return flag, label
    def image(self, i):
        _, label, rest = self.header_full(i)
        return label, Image.open(io.BytesIO(rest)).convert("RGB")

class MXRecDataset(Dataset):
    def __init__(self, rec, idx, transform):
        self.r = RecReader(rec, idx); self.transform = transform
        flag0, lab0, _ = self.r.header_full(0)
        if flag0 > 0 and len(lab0) >= 1:      # standard insightface header record
            self.imgidx = list(range(1, int(lab0[0])))
        else:
            self.imgidx = [k for k in self.r.offsets if k != 0]
        self.labels = [int(self.r.label_only(i)[1]) for i in self.imgidx]
        self.num_classes = int(max(self.labels)) + 1
        self.r.f = None                       # let workers reopen after fork
    def __len__(self): return len(self.imgidx)
    def __getitem__(self, k):
        _, im = self.r.image(self.imgidx[k])
        return self.transform(im), self.labels[k]

train_tf = transforms.Compose([transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)])
eval_tf  = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)])

print("indexing dataset (reading headers)...")
train_ds = MXRecDataset(REC, IDX, train_tf)
NUM_CLASSES = train_ds.num_classes
print("images:", len(train_ds), "| identities:", NUM_CLASSES)
# sanity: decode one image
_lab, _im = train_ds.r.image(train_ds.imgidx[0]); print("sample image size:", _im.size, "label:", int(_lab))

NW = min(4, os.cpu_count() or 2)
train_loader = DataLoader(train_ds, batch_size=CFG["BATCH"], shuffle=True, num_workers=NW,
                          pin_memory=True, drop_last=True, persistent_workers=True)
print("batches/epoch:", len(train_loader))


indexing dataset (reading headers)...
images: 490623 | identities: 10572
sample image size: (112, 112) label: 0
batches/epoch: 1916


## 4 · MobileFaceNet + ArcFace
Canonical MobileFaceNet (~1.0 M params, 128-d embedding) with additive-angular-margin loss.

In [5]:
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(s,i,o,k,st,p,g=1,act=True):
        super().__init__(); s.c=nn.Conv2d(i,o,k,st,p,groups=g,bias=False); s.b=nn.BatchNorm2d(o)
        s.a=nn.PReLU(o) if act else nn.Identity()
    def forward(s,x): return s.a(s.b(s.c(x)))

class IRes(nn.Module):
    def __init__(s,i,o,st,e):
        super().__init__(); h=i*e; s.res=(st==1 and i==o)
        s.m=nn.Sequential(ConvBlock(i,h,1,1,0),ConvBlock(h,h,3,st,1,g=h),
                          nn.Conv2d(h,o,1,1,0,bias=False),nn.BatchNorm2d(o))
    def forward(s,x): y=s.m(x); return x+y if s.res else y

class MobileFaceNet(nn.Module):
    def __init__(s,emb=128):
        super().__init__()
        s.conv1=ConvBlock(3,64,3,2,1); s.conv2=ConvBlock(64,64,3,1,1,g=64)
        L=[]; inc=64
        for t,c,nr,st in [(2,64,5,2),(4,128,1,2),(2,128,6,1),(4,128,1,2),(2,128,2,1)]:
            for j in range(nr): L.append(IRes(inc,c,st if j==0 else 1,t)); inc=c
        s.blocks=nn.Sequential(*L)
        s.sep=ConvBlock(128,512,1,1,0); s.gd=ConvBlock(512,512,7,1,0,g=512,act=False)
        s.out=nn.Conv2d(512,emb,1,1,0,bias=False); s.bn=nn.BatchNorm1d(emb)
    def forward(s,x):
        x=s.conv2(s.conv1(x)); x=s.sep(s.blocks(x)); x=s.gd(x)
        return s.bn(s.out(x).flatten(1))

class ArcMargin(nn.Module):
    def __init__(s,i,o,sc=64.0,m=0.50):
        super().__init__(); s.s=sc; s.cm=math.cos(m); s.sm=math.sin(m)
        s.th=math.cos(math.pi-m); s.mm=math.sin(math.pi-m)*m
        s.w=nn.Parameter(torch.empty(o,i)); nn.init.xavier_normal_(s.w)
    def forward(s,emb,lab):
        cos=F.linear(F.normalize(emb),F.normalize(s.w)).clamp(-1+1e-7,1-1e-7)
        sin=torch.sqrt((1-cos*cos).clamp_min(1e-9)); phi=cos*s.cm-sin*s.sm
        phi=torch.where(cos>s.th,phi,cos-s.mm)
        oh=F.one_hot(lab,cos.size(1)).float()
        return s.s*(oh*phi+(1-oh)*cos)

model=MobileFaceNet(CFG["EMB_SIZE"]).to(DEVICE)
head =ArcMargin(CFG["EMB_SIZE"],NUM_CLASSES,CFG["ARC_S"],CFG["ARC_M"]).to(DEVICE)
print("params:",sum(p.numel() for p in model.parameters()))


params: 1003136


## 5 · Real LFW evaluation (from `eval/lfw.bin`)
Decodes the bundled LFW pairs, embeds each image plus its horizontal flip, and reports standard 10-fold
verification accuracy with cosine similarity — the figure judges expect for the ">95%" criterion.

In [6]:
import pickle
from sklearn.model_selection import KFold

def _load_bin(path):
    bins, issame = pickle.load(open(path, "rb"), encoding="bytes")
    imgs = []
    for b in bins:
        bb = b if isinstance(b, (bytes, bytearray)) else bytes(b)
        imgs.append(eval_tf(Image.open(io.BytesIO(bb)).convert("RGB")))
    return torch.stack(imgs), np.array(issame, dtype=bool)

_LFW_CACHE = {}
@torch.no_grad()
def lfw_accuracy(net, path, bs=256):
    if path not in _LFW_CACHE: _LFW_CACHE[path] = _load_bin(path)
    X, issame = _LFW_CACHE[path]; net.eval()
    embs = []
    for i in range(0, len(X), bs):
        xb = X[i:i+bs].to(DEVICE)
        e = net(xb) + net(torch.flip(xb, [3]))      # flip augmentation
        embs.append(F.normalize(e).cpu())
    E = torch.cat(embs).numpy(); e1, e2 = E[0::2], E[1::2]
    sims = (e1 * e2).sum(1)
    thr = np.arange(-1, 1, 0.005); accs = []
    for tr, te in KFold(n_splits=min(10,len(issame)), shuffle=False).split(np.arange(len(issame))):
        bt = max(thr, key=lambda t: ((sims[tr] > t) == issame[tr]).mean())
        accs.append(((sims[te] > bt) == issame[te]).mean())
    return float(np.mean(accs))

if LFW_BIN: print("LFW pairs loaded; baseline (untrained) acc:", round(lfw_accuracy(model, LFW_BIN)*100,2), "%")
else: print("No lfw.bin found — training will still run; attach eval/*.bin for the LFW metric.")


LFW pairs loaded; baseline (untrained) acc: 66.03 %


## 6 · Train (time-budgeted, AMP) — best checkpoint by LFW accuracy

In [7]:
from tqdm.auto import tqdm
params=list(model.parameters())+list(head.parameters())
opt=torch.optim.SGD(params,lr=CFG["LR"],momentum=CFG["MOMENTUM"],weight_decay=CFG["WEIGHT_DECAY"])
total=CFG["MAX_EPOCHS"]*len(train_loader); scaler=torch.cuda.amp.GradScaler(); crit=nn.CrossEntropyLoss()
def lr_at(it):
    if it<CFG["WARMUP_ITERS"]: return CFG["LR"]*it/max(1,CFG["WARMUP_ITERS"])
    p=(it-CFG["WARMUP_ITERS"])/max(1,total-CFG["WARMUP_ITERS"]); return 0.5*CFG["LR"]*(1+math.cos(math.pi*min(1.0,p)))

t0=time.time(); budget=CFG["TIME_BUDGET_HOURS"]*3600; best=0.0; it=0; stop=False
for ep in range(CFG["MAX_EPOCHS"]):
    model.train(); head.train(); run=0.0
    pb=tqdm(train_loader,desc=f"epoch {ep+1}/{CFG['MAX_EPOCHS']}")
    for xb,yb in pb:
        for g in opt.param_groups: g["lr"]=lr_at(it)
        xb,yb=xb.to(DEVICE,non_blocking=True),yb.to(DEVICE,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(): loss=crit(head(model(xb),yb),yb)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        run+=loss.item(); it+=1; pb.set_postfix(loss=f"{run/(pb.n+1):.3f}",lr=f"{opt.param_groups[0]['lr']:.4f}")
        if time.time()-t0>budget: print("\n[time budget reached]"); stop=True; break
    acc = lfw_accuracy(model, LFW_BIN) if LFW_BIN else 0.0
    print(f"epoch {ep+1}: LFW acc={acc*100:.2f}%  elapsed {(time.time()-t0)/3600:.2f}h")
    if acc>=best:
        best=acc; torch.save(model.state_dict(),os.path.join(CFG["OUT_DIR"],"mobilefacenet_fp32.pt"))
        print(f"  saved best ({best*100:.2f}%)")
    if stop: break
print(f"\nBEST LFW accuracy: {best*100:.2f}%")


/tmp/ipykernel_58/2612895400.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  total=CFG["MAX_EPOCHS"]*len(train_loader); scaler=torch.cuda.amp.GradScaler(); crit=nn.CrossEntropyLoss()


epoch 1/40:   0%|          | 0/1916 [00:00<?, ?it/s]

/tmp/ipykernel_58/2612895400.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): loss=crit(head(model(xb),yb),yb)


epoch 1: LFW acc=85.90%  elapsed 0.16h
  saved best (85.90%)


epoch 2/40:   0%|          | 0/1916 [00:00<?, ?it/s]

...

epoch 38: LFW acc=99.08%  elapsed 6.25h


epoch 39/40:   0%|          | 0/1916 [00:00<?, ?it/s]

epoch 39: LFW acc=99.08%  elapsed 6.41h


epoch 40/40:   0%|          | 0/1916 [00:00<?, ?it/s]

epoch 40: LFW acc=99.15%  elapsed 6.58h

BEST LFW accuracy: 99.28%


## 7 · Load best & export self-contained ONNX (FP32) + parity
`dynamo=False` keeps weights inline in one file (mobile-friendly).

In [8]:
import onnx, onnxruntime as ort
model.load_state_dict(torch.load(os.path.join(CFG["OUT_DIR"],"mobilefacenet_fp32.pt"),map_location=DEVICE))
model.eval().cpu()
onnx_fp32=os.path.join(CFG["OUT_DIR"],"mobilefacenet_fp32.onnx"); dummy=torch.randn(1,3,112,112)
torch.onnx.export(model,dummy,onnx_fp32,input_names=["input"],output_names=["embedding"],
                  dynamic_axes={"input":{0:"N"},"embedding":{0:"N"}},opset_version=13,dynamo=False)
onnx.checker.check_model(onnx.load(onnx_fp32))
sess=ort.InferenceSession(onnx_fp32,providers=["CPUExecutionProvider"])
with torch.no_grad(): ref=model(dummy).numpy()
out=sess.run(None,{"input":dummy.numpy()})[0]
print("FP32 ONNX MB:",round(os.path.getsize(onnx_fp32)/1e6,3)," max|torch-onnx|:",float(np.abs(ref-out).max()))
model.to(DEVICE)


/tmp/ipykernel_58/4021050584.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,dummy,onnx_fp32,input_names=["input"],output_names=["embedding"],


FP32 ONNX MB: 3.997  max|torch-onnx|: 9.5367431640625e-07


MobileFaceNet(
  (conv1): ConvBlock(
    (c): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (b): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (a): PReLU(num_parameters=64)
  )
  (conv2): ConvBlock(
    (c): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
    (b): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (a): PReLU(num_parameters=64)
  )
  (blocks): Sequential(
    (0): IRes(
      (m): Sequential(
        (0): ConvBlock(
          (c): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (b): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (a): PReLU(num_parameters=128)
        )
        (1): ConvBlock(
          (c): Conv2d(128, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=128, bias=False)
          (b): BatchNorm2d(128, eps=1e-05, momentum=0.1, affi

## 8 · INT8 quantization (ONNX)

In [9]:
from onnxruntime.quantization import quantize_dynamic, QuantType
onnx_int8=os.path.join(CFG["OUT_DIR"],"mobilefacenet_int8.onnx")
quantize_dynamic(onnx_fp32,onnx_int8,weight_type=QuantType.QInt8)
print("INT8 ONNX MB:",round(os.path.getsize(onnx_int8)/1e6,3))


INT8 ONNX MB: 1.149


## 9 · (Optional) TFLite export — for `react-native-fast-tflite`

In [10]:
try:
    pip("tensorflow-cpu","onnx2tf","onnx_graphsurgeon","sng4onnx","onnxsim","ai-edge-litert")
    import onnx2tf
    tfl=os.path.join(CFG["OUT_DIR"],"tflite")
    onnx2tf.convert(input_onnx_file_path=onnx_fp32,output_folder_path=tfl,
                    output_integer_quantized_tflite=True,non_verbose=True)
    for f in sorted(glob.glob(os.path.join(tfl,"*.tflite"))):
        print(os.path.basename(f),"->",round(os.path.getsize(f)/1e6,3),"MB")
except Exception as e:
    print("TFLite skipped (use ONNX):",repr(e))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 14.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 16.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.0/223.0 kB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 17.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 15.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 16.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 17.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.3/222.3 kB 13.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.3/222.3 kB 16.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-cloud-spanner 3.64.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.0 which is incompatible.
google-cloud-bigtable 2.36.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.0 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.0 which is incompatible.
google-cloud-firestore 2.26

TFLite skipped (use ONNX): NotFoundError()


## 10 · ✅ Constraints verification

In [11]:
import onnxruntime as ort, numpy as np, time
def size_mb(p): return os.path.getsize(p)/1e6
fp32_mb=size_mb(onnx_fp32); int8_mb=size_mb(onnx_int8)
so=ort.SessionOptions(); so.intra_op_num_threads=1
s=ort.InferenceSession(onnx_int8,sess_options=so,providers=["CPUExecutionProvider"])
x=np.random.randn(1,3,112,112).astype("float32")
for _ in range(10): s.run(None,{"input":x})
ts=[]
for _ in range(100):
    t=time.perf_counter(); s.run(None,{"input":x}); ts.append(time.perf_counter()-t)
lat_ms=float(np.median(ts))*1000
print("Constraint checks")
print(f"  footprint   : FP32 {fp32_mb:.2f} MB / INT8 {int8_mb:.2f} MB   target < 20 MB   -> {'PASS' if max(fp32_mb,int8_mb)<20 else 'FAIL'}")
print(f"  latency/face: {lat_ms:.2f} ms (1 thread)                    target < 1000 ms -> {'PASS' if lat_ms<1000 else 'FAIL'}")
print(f"  LFW accuracy: {best*100:.2f}%                                target > 95%     -> {'PASS' if best>0.95 else 'train longer / fine-tune'}")
print(f"  open-source : MobileFaceNet+ArcFace trained by you           -> PASS")
assert max(fp32_mb,int8_mb)<20 and lat_ms<1000


Constraint checks
  footprint   : FP32 4.00 MB / INT8 1.15 MB   target < 20 MB   -> PASS
  latency/face: 63.03 ms (1 thread)                    target < 1000 ms -> PASS
  LFW accuracy: 99.28%                                target > 95%     -> PASS
  open-source : MobileFaceNet+ArcFace trained by you           -> PASS


## 11 · Artifacts

In [12]:
print("Saved to /kaggle/working/artifacts (Output tab):")
for f in sorted(glob.glob(os.path.join(CFG["OUT_DIR"],"**","*"),recursive=True)):
    if os.path.isfile(f): print(f"  {os.path.relpath(f,CFG['OUT_DIR']):40s} {size_mb(f):7.3f} MB")


Saved to /kaggle/working/artifacts (Output tab):
  mobilefacenet_fp32.onnx                    3.997 MB
  mobilefacenet_fp32.pt                      4.206 MB
  mobilefacenet_int8.onnx                    1.149 MB


## Notes — before submitting

**Indian demographics (judging criterion).** CASIA skews Western/East-Asian. Fine-tune a few epochs from
`mobilefacenet_fp32.pt` on Indian faces (Indian Movie Face DB or your own captures), same 112×112 alignment.

**App-side preprocessing (must match exactly).** Aligned 112×112 RGB → `(pixel/255-0.5)/0.5` → NCHW → run model →
**L2-normalise** the 128-d output → cosine-compare to the enrolment threshold.

**Other `eval/*.bin` sets.** `lfw_accuracy(model, path)` works on any of them — try `cfp_fp.bin` (pose) and
`agedb_30.bin` (age) for extra robustness evidence in your report.

**This is the recognition model only.** Detector/aligner + liveness (MediaPipe landmarks / MiniFASNet) are separate
pre-trained pieces and need no training here.